[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/56_mamba_ssm_solution.ipynb)

# 🔴 Solution: Mamba SSM Step

Reference solution for a single Mamba selective state-space model recurrent step.

$$h_t = \overline{A} \cdot h_{t-1} + \overline{B} \cdot x_t, \quad y_t = C \cdot h_t$$

where $\overline{A} = \exp(\Delta \cdot A)$ and $\overline{B} = \Delta \cdot B$.

In [ ]:
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def mamba_ssm_step(x, h, A, B, C, delta):
    dA = torch.exp(delta.unsqueeze(-1) * A.unsqueeze(0))
    dB = delta.unsqueeze(-1) * B
    h_new = dA * h + dB * x.unsqueeze(-1)
    y = (h_new * C).sum(dim=-1)
    return y, h_new

In [ ]:
torch.manual_seed(0)
B_size, D, N = 2, 8, 4  # batch, channels, state_dim

x     = torch.randn(B_size, D)
h     = torch.zeros(B_size, D, N)
A     = -torch.rand(D, N)          # negative for stability
B_mat = torch.randn(D, N)
C     = torch.randn(B_size, D, N)
delta = torch.rand(B_size, D).add(0.1)  # positive step sizes

y, h_new = mamba_ssm_step(x, h, A, B_mat, C, delta)

print("y shape:    ", y.shape)      # (2, 8)
print("h_new shape:", h_new.shape)  # (2, 8, 4)

# Verify dA = exp(delta * A) numerically for first element
dA_manual = torch.exp(delta[0, 0] * A[0])
dA_check  = torch.exp(delta[0:1, 0:1] * A[0:1])[0, 0]
print("dA formula check (should be ~0):", (dA_manual - dA_check).abs().max().item())

In [ ]:
from torch_judge import check
check("mamba_ssm")